In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformer_lens import HookedTransformer
import time
import os
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [ ]:
model_name = "Qwen2.5-7B-Instruct"
dataset = "allenai/WildChat-1M"
Y_lengths = torch.load(f"data/pred_gen/{dataset}/{model_name}/Y_lengths.pt")

torch.Size([3200])


In [ ]:
Y_lengths_2048 = Y_lengths.clone()
Y_lengths_2048[Y_lengths_2048 == -1] = 2048
q_25, q_50, q_75 = torch.quantile(Y_lengths_2048.float(), torch.tensor([0.25, 0.5, 0.75]))
q = q_50
Y_binray = Y_lengths_2048 >= q
X = torch.load(f"data/pred_gen/{dataset}/{model_name}/layer_{2}/X_last.pt")
X = X[:Y_binray.shape[0]]  # align sizes
X_train, X_test, Y_train, Y_test = train_test_split(X, Y_binray, test_size=0.2, random_state=42)
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, Y_train)
accuracy = clf.score(X_test, Y_test)
print(f"Accuracy at layer 2: {accuracy:.4f}")



tensor(2142)
